# Qwen3.5-4B Telecom RCA: SFT + Held-Out Evaluation

This Colab notebook fine-tunes `unsloth/Qwen3.5-4B` with 16-bit LoRA on the synthetic telecom RCA reasoning trajectories, evaluates held-out loss during training, and measures generated-label accuracy on all 864 official validation questions.

Key rules:

- The synthetic reasoning is trained in Qwen3.5's native `<think>...</think>` format.
- The validation set contains only known final labels. No validation reasoning is invented.
- Validation examples are never optimizer inputs.
- Qwen3.5 4-bit QLoRA is intentionally disabled because Unsloth currently recommends 16-bit LoRA for this model family.
- GRPO is a separate stage and must reuse this exact base model, tokenizer/chat template, and response contract.


In [3]:
import os
import sys

print("Python:", sys.executable)

# Install uv using the notebook's actual Python.
# Use only one -q, not -qqq.
!{sys.executable} -m pip install -q --upgrade uv

# Make every uv command install into the notebook environment.
os.environ["UV_SYSTEM_PYTHON"] = "1"

Python: /usr/local/bin/python

[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


## 1. Install the Qwen3.5-compatible Unsloth stack

Use a GPU runtime: **Runtime → Change runtime type → T4 GPU**. The first run compiles Qwen3.5's hybrid-model kernels and can take several minutes.


In [4]:
!uv pip install --upgrade \
    "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo.git" \
    "unsloth[base] @ git+https://github.com/unslothai/unsloth.git" \
    bitsandbytes \
    "xformers==0.0.32.post2" \
    datasets \
    pandas

Using Python 3.12.6 environment at: /usr/local
Resolved 102 packages in 13.94s
Prepared 74 packages in 13.21s
Uninstalled 53 packages in 3.06s
Installed 74 packages in 614ms
 - accelerate==1.10.1
 + accelerate==1.14.0
 - aiohappyeyeballs==2.4.3
 + aiohappyeyeballs==2.7.1
 - aiohttp==3.10.8
 + aiohttp==3.14.3
 - aiosignal==1.3.1
 + aiosignal==1.4.0
 + annotated-doc==0.0.4
 - anyio==4.10.0
 + anyio==4.14.2
 - attrs==24.2.0
 + attrs==26.1.0
 + bitsandbytes==0.49.2
 - certifi==2024.8.30
 + certifi==2026.7.22
 - charset-normalizer==3.4.3
 + charset-normalizer==3.4.9
 - click==8.2.1
 + click==8.4.2
 + cut-cross-entropy==25.1.1
 + datasets==4.3.0
 - diffusers==0.35.1
 + diffusers==0.39.0
 + dill==0.4.0
 + docstring-parser==0.18.0
 - filelock==3.13.1
 + filelock==3.32.0
 - frozenlist==1.4.1
 + frozenlist==1.8.0
 - fsspec==2024.6.1
 + fsspec==2025.9.0
 + hf-transfer==0.1.9
 - hf-xet==1.1.9
 + hf-xet==1.5.2
 - huggingface-hub==0.34.4
 + huggingface-hub==1.24.0
 - idna==3.10
 + idna==3.18
 - impo

In [5]:
!uv pip install --upgrade --no-deps \
    "transformers==5.2.0" \
    "tokenizers>=0.22.0,<=0.23.0" \
    "trl==0.22.2" \
    "torchao>=0.16.0"

Using Python 3.12.6 environment at: /usr/local
Resolved 4 packages in 115ms
Prepared 2 packages in 678ms
Uninstalled 2 packages in 121ms
Installed 2 packages in 110ms
 - transformers==5.5.0
 + transformers==5.2.0
 - trl==0.24.0
 + trl==0.22.2


In [6]:
from importlib.metadata import version, PackageNotFoundError

packages = [
    "torch",
    "transformers",
    "trl",
    "unsloth",
    "unsloth_zoo",
    "tokenizers",
    "xformers",
    "bitsandbytes",
    "torchao",
]

for package in packages:
    try:
        print(f"{package:25s}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package:25s}: NOT INSTALLED")

torch                    : 2.8.0+cu129
transformers             : 5.2.0
trl                      : 0.22.2
unsloth                  : 2026.7.5
unsloth_zoo              : 2026.7.6
tokenizers               : 0.22.2
xformers                 : 0.0.32.post2
bitsandbytes             : 0.49.2
torchao                  : 0.17.0


## 2. Configuration and input files

Upload these two files into the Colab working directory:

- `sft_train_data.jsonl` — the generated synthetic `question`/`response` data.
- `sft_validation_data.jsonl` — the prepared 864-row held-out validation artifact.

The notebook also checks `/content/drive/MyDrive/` if the files are not present locally.


In [7]:
import json
import os
import random
import re
from collections import Counter
from pathlib import Path

import numpy as np
import torch
from datasets import Dataset

SEED = 42
MODEL_NAME = "unsloth/Qwen3.5-4B"
MAX_SEQ_LENGTH = 8192
OUTPUT_DIR = "qwen35_4b_sft_outputs"
ADAPTER_DIR = "qwen35_4b_sft_lora"
GENERATION_MAX_NEW_TOKENS = 1024

# Set to a positive integer for a quick smoke test. Keep None for the official
# all-864-question evaluation.
EVAL_LIMIT = 10

SYSTEM_PROMPT = (
    "You are a senior telecom root-cause analysis engineer. Analyze the supplied "
    "drive-test and engineering evidence carefully. Follow the candidate identifiers "
    "defined in the user prompt and finish with exactly one selected identifier "
    "enclosed in \\boxed{}."
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

assert torch.cuda.is_available(), "A CUDA GPU runtime is required."
print("GPU:", torch.cuda.get_device_name(0))
print("BF16 supported:", torch.cuda.is_bf16_supported())


def locate_file(filename):
    candidates = [
        Path(filename),
        Path("/root") / filename,
        Path("/root") / filename,
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f"Could not find {filename}. Upload it to Colab or place it in MyDrive."
    )


TRAIN_PATH = locate_file("sft_train_data.jsonl")
VALIDATION_PATH = locate_file("sft_validation_data.jsonl")
print("Training data:", TRAIN_PATH)
print("Validation data:", VALIDATION_PATH)

GPU: NVIDIA A100-SXM4-40GB
BF16 supported: True
Training data: sft_train_data.jsonl
Validation data: sft_validation_data.jsonl


## 3. Load Qwen3.5-4B and attach language-only LoRA

`fast_inference=False` is deliberate: current Qwen3.5 reinforcement learning and training should use Unsloth inference rather than the older vLLM path. Vision parameters remain frozen because this task is text-only.


In [8]:
from unsloth import FastLanguageModel
import torch

# Enable meta-data support to bypass potential torch.nonzero() errors on certain kernels
torch.fx.experimental._config.meta_nonzero_assume_all_nonzero = True

# Load model in 4-bit to fit within T4 GPU memory (16GB)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    full_finetuning=False,
    fast_inference=False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    max_seq_length=MAX_SEQ_LENGTH,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Loaded 4-bit QLoRA:", MODEL_NAME)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: OpenAI failed to import - ignoring for now.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.5: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu129. CUDA: 8.0. CUDA Toolkit: 12.9. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.
Loaded 4-bit QLoRA: unsloth/Qwen3.5-4B


## 4. Validate and format the datasets

Training responses are converted from their existing four-section form into:

```text
<think>
...synthetic reasoning...
</think>

\boxed{R#}
```

Validation responses remain answer-only (`\boxed{C#}`). They support completion-only held-out loss without fabricating reasoning that is absent from the source data.


In [9]:
TRAIN_BOX_RE = re.compile(r"\\boxed\{(R[1-8])\}[.!?;:]*\s*$")
VALID_RESPONSE_RE = re.compile(r"\\boxed\{(C[1-8])\}")


def load_jsonl(path):
    records = []
    with path.open(encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, 1):
            if not line.strip():
                raise ValueError(f"{path}:{line_number} is empty")
            try:
                item = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(f"{path}:{line_number}: {error}") from error
            if not isinstance(item, dict):
                raise ValueError(f"{path}:{line_number} must be a JSON object")
            records.append(item)
    return records


def format_training_response(response, location):
    if not isinstance(response, str) or not response.strip():
        raise ValueError(f"{location}: response must be non-empty")
    matches = list(TRAIN_BOX_RE.finditer(response))
    all_boxes = re.findall(r"\\boxed\{R[1-8]\}", response)
    if len(matches) != 1 or len(all_boxes) != 1:
        raise ValueError(f"{location}: expected exactly one terminal boxed R1-R8 label")
    match = matches[0]
    reasoning = response[: match.start()].rstrip()
    if not reasoning:
        raise ValueError(f"{location}: reasoning is empty")
    return f"<think>\n{reasoning}\n</think>\n\n\\boxed{{{match.group(1)}}}"


def conversation(question, assistant_response):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
        {"role": "assistant", "content": assistant_response},
    ]


raw_train = load_jsonl(TRAIN_PATH)
raw_validation = load_jsonl(VALIDATION_PATH)
assert len(raw_train) == 1941, f"Expected 1,941 SFT examples; got {len(raw_train)}"
assert len(raw_validation) == 864, (
    f"Expected 864 validation examples; got {len(raw_validation)}"
)

train_rows = []
train_questions = set()
for index, item in enumerate(raw_train):
    if set(item) != {"question", "response"}:
        raise ValueError(f"train row {index}: expected only question and response")
    question = item["question"]
    if (
        not isinstance(question, str)
        or not question.strip()
        or question in train_questions
    ):
        raise ValueError(f"train row {index}: empty or duplicate question")
    response = format_training_response(item["response"], f"train row {index}")
    train_rows.append({"messages": conversation(question, response)})
    train_questions.add(question)

validation_rows = []
validation_questions = set()
validation_labels = Counter()
for index, item in enumerate(raw_validation):
    if set(item) != {"id", "question", "response"}:
        raise ValueError(f"validation row {index}: expected id, question, and response")
    identifier = item["id"]
    question = item["question"]
    response = item["response"]
    match = VALID_RESPONSE_RE.fullmatch(response)
    if not identifier or not question or not match:
        raise ValueError(f"validation row {index}: malformed id, question, or response")
    if question in validation_questions or question in train_questions:
        raise ValueError(
            f"validation row {index}: duplicate or train/validation overlap"
        )
    label = match.group(1)
    validation_rows.append(
        {
            "id": identifier,
            "question": question,
            "target": label,
            "messages": conversation(question, response),
        }
    )
    validation_questions.add(question)
    validation_labels[label] += 1

expected_balance = Counter({f"C{i}": 108 for i in range(1, 9)})
assert validation_labels == expected_balance, validation_labels
print(
    f"Validated {len(train_rows):,} SFT and {len(validation_rows):,} held-out examples"
)
print("Validation balance:", dict(sorted(validation_labels.items())))
print("Example formatted response tail:")
print(train_rows[0]["messages"][-1]["content"][-400:])

Validated 1,941 SFT and 864 held-out examples
Validation balance: {'C1': 108, 'C2': 108, 'C3': 108, 'C4': 108, 'C5': 108, 'C6': 108, 'C7': 108, 'C8': 108}
Example formatted response tail:
evice switches cells three times in four seconds, triggering continuous connection reconfiguration and radio interruption that suppresses throughput below 600 Mbps. All other potential causes are mathematically or empirically excluded based on colocation verification, distance thresholds, tilt/beamwidth relationships, PCI modulo checks, speed limits, and RB scheduling metrics.
</think>

\boxed{R1}


In [10]:
def render_manual(messages):
    # Manually construct the prompt to avoid chat template image-parsing side effects
    res = ""
    for m in messages:
        res += f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>\n"
    return res


train_text_rows = [{"text": render_manual(row["messages"])} for row in train_rows]
validation_text_rows = [
    {"text": render_manual(row["messages"])} for row in validation_rows
]
train_dataset = Dataset.from_list(train_text_rows)
validation_dataset = Dataset.from_list(validation_text_rows)


def token_length(text):
    # Access the tokenizer within the processor to avoid multimodal logic
    return len(tokenizer.tokenizer.encode(text, add_special_tokens=False))


train_lengths = [token_length(row["text"]) for row in train_text_rows]
validation_lengths = [token_length(row["text"]) for row in validation_text_rows]
max_observed = max(train_lengths + validation_lengths)

if max_observed > MAX_SEQ_LENGTH:
    raise ValueError(
        f"Longest example is {max_observed} tokens, exceeding MAX_SEQ_LENGTH."
    )

for name, lengths in (("train", train_lengths), ("validation", validation_lengths)):
    print(
        f"{name}: min={min(lengths)}, median={int(np.median(lengths))}, p95={int(np.percentile(lengths, 95))}, max={max(lengths)} tokens"
    )

probe = train_text_rows[0]["text"]
assert "<|im_start|>assistant\n" in probe
assert "<think>\n" in probe and "\\boxed{" in probe

train: min=2745, median=3343, p95=3795, max=4278 tokens
validation: min=2116, median=2439, p95=2765, max=2941 tokens


## 5. Supervised fine-tuning

The trainer masks every token before the assistant response. Consequently, validation loss is computed on the gold boxed answer rather than on the long question text. Packing is disabled so examples and masks cannot cross sequence boundaries.


In [11]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only
from transformers import DataCollatorForSeq2Seq
import gc

# Clear VRAM before initialization
torch.cuda.empty_cache()
gc.collect()

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    dataset_num_proc=1,
    packing=False,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=5,
    learning_rate=2e-4,
    warmup_steps=15,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    seed=SEED,
    report_to="none",
)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer.tokenizer, model=model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer.tokenizer,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    args=training_args,
    data_collator=data_collator,
)

# Mask prompt so model only learns from the <think> and \boxed{} parts
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/1941 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/864 [00:00<?, ? examples/s]

[accelerate.utils.other|WARNING][RANK 0] Detected kernel version 4.19.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Map:   0%|          | 0/1941 [00:00<?, ? examples/s]

Map:   0%|          | 0/864 [00:00<?, ? examples/s]

In [ ]:
!nvidia-smi --query-gpu=timestamp,name,utilization.gpu,utilization.memory,memory.used,power.draw --format=csv -l 1

In [12]:
trainer_stats = trainer.train()

print(trainer_stats)
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best held-out loss:", trainer.state.best_metric)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,941 | Num Epochs = 5 | Total steps = 610
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 21,233,664 of 4,560,499,200 (0.47% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Epoch,Training Loss,Validation Loss
1,0.425406,3.737164
2,0.372397,3.694749
3,0.336926,3.684375
4,0.310671,3.720578
5,0.278275,3.817038


Unsloth: Restored added_tokens_decoder metadata in qwen35_4b_sft_outputs/checkpoint-122/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in qwen35_4b_sft_outputs/checkpoint-244/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in qwen35_4b_sft_outputs/checkpoint-366/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in qwen35_4b_sft_outputs/checkpoint-488/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in qwen35_4b_sft_outputs/checkpoint-610/tokenizer_config.json.


TrainOutput(global_step=610, training_loss=0.3672722785199275, metrics={'train_runtime': 10242.3229, 'train_samples_per_second': 0.948, 'train_steps_per_second': 0.06, 'total_flos': 8.273674074840545e+17, 'train_loss': 0.3672722785199275, 'epoch': 5.0})
Best checkpoint: qwen35_4b_sft_outputs/checkpoint-366
Best held-out loss: 3.684375286102295


In [13]:
torch.cuda.empty_cache()
import gc

gc.collect()

0

## 6. Full generated-label evaluation

This is the primary held-out metric. The model receives only the system prompt and question, generates its own reasoning and final answer, and receives credit only when the final non-whitespace text is a boxed `C1`–`C8` label.

Running all 864 long prompts on a T4 can take substantial time. Set `EVAL_LIMIT` in Section 2 only for debugging; leave it as `None` for the official result.


In [17]:
EVAL_LIMIT = None

In [ ]:
import re
import torch
from collections import defaultdict
from tqdm.auto import tqdm

# Enable Unsloth inference mode
FastLanguageModel.for_inference(model)
model.eval()

# Your tokenizer appears to be a processor/wrapper.
text_tokenizer = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer

# Required for batched generation with decoder-only models.
text_tokenizer.padding_side = "left"

# Some causal LMs do not define a separate padding token.
if text_tokenizer.pad_token_id is None:
    text_tokenizer.pad_token = text_tokenizer.eos_token

# Keep wrapper settings synchronized when possible.
if hasattr(tokenizer, "padding_side"):
    tokenizer.padding_side = "left"

if hasattr(tokenizer, "pad_token_id") and tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# Keep model generation settings synchronized.
model.generation_config.pad_token_id = text_tokenizer.pad_token_id
model.generation_config.eos_token_id = text_tokenizer.eos_token_id

print("Padding side:", text_tokenizer.padding_side)
print("Pad token:", text_tokenizer.pad_token)
print("Pad token ID:", text_tokenizer.pad_token_id)
print("EOS token ID:", text_tokenizer.eos_token_id)


STRICT_FINAL_BOX_RE = re.compile(r"\\boxed\s*\{\s*(C[1-8])\s*\}")

evaluation_records = (
    validation_rows if EVAL_LIMIT is None else validation_rows[:EVAL_LIMIT]
)

predictions = []

BATCH_SIZE = 16

for i in tqdm(
    range(0, len(evaluation_records), BATCH_SIZE),
    desc="Batched Validation",
):
    batch = evaluation_records[i : i + BATCH_SIZE]

    prompts = []

    for row in batch:
        prompt_messages = [
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": row["question"],
            },
        ]

        prompt = tokenizer.apply_chat_template(
            prompt_messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=True,
        )

        prompts.append(prompt)

    # Left-padded batched tokenization
    inputs = text_tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        add_special_tokens=False,
    ).to(model.device)

    # All sequences have this padded input width.
    prompt_width = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=GENERATION_MAX_NEW_TOKENS,
            use_cache=True,
            do_sample=False,
            pad_token_id=text_tokenizer.pad_token_id,
            eos_token_id=text_tokenizer.eos_token_id,
        )

    # Remove the entire padded prompt portion.
    generated_ids = output_ids[:, prompt_width:]

    completions = text_tokenizer.batch_decode(
        generated_ids,
        skip_special_tokens=True,
    )

    for row, completion in zip(batch, completions):
        completion = completion.strip()

        # Use the final boxed label if the response contains more than one.
        matches = STRICT_FINAL_BOX_RE.findall(completion)
        prediction = matches[-1] if matches else None

        predictions.append(
            {
                "id": row["id"],
                "target": row["target"],
                "prediction": prediction,
                "correct": prediction == row["target"],
                "format_valid": prediction is not None,
                "completion": completion,
            }
        )

correct = sum(record["correct"] for record in predictions)
total = len(predictions)

accuracy = correct / total if total else 0.0

print(f"Exact generated-label accuracy: {accuracy:.2%} ({correct}/{total})")

Padding side: left
Pad token: <|vision_pad|>
Pad token ID: 248055
EOS token ID: 248046


Batched Validation:   0%|          | 0/54 [00:00<?, ?it/s]

In [ ]:
import json
import pandas as pd

prediction_frame = pd.DataFrame(predictions)

labels = [f"C{i}" for i in range(1, 9)]

# Calculate summary metrics here so they always exist
total = len(prediction_frame)
correct = int(prediction_frame["correct"].sum())
accuracy = correct / total if total > 0 else 0.0

format_valid = int(prediction_frame["format_valid"].sum())
format_valid_rate = format_valid / total if total > 0 else 0.0

confusion = pd.crosstab(
    prediction_frame["target"],
    prediction_frame["prediction"].fillna("MALFORMED"),
    dropna=False,
).reindex(
    index=labels,
    columns=labels + ["MALFORMED"],
    fill_value=0,
)

# Reindex ensures every C1-C8 class appears, even if a class has no examples
per_class = (
    prediction_frame.groupby("target", sort=True)["correct"]
    .agg(["sum", "count", "mean"])
    .rename(
        columns={
            "sum": "correct",
            "mean": "accuracy",
        }
    )
    .reindex(labels)
)

# Replace missing counts with zero
per_class["correct"] = per_class["correct"].fillna(0).astype(int)
per_class["count"] = per_class["count"].fillna(0).astype(int)
per_class["accuracy"] = per_class["accuracy"].fillna(0.0)

display(per_class)
display(confusion)

print(f"Accuracy: {accuracy:.2%} ({correct}/{total})")
print(f"Valid output format: {format_valid_rate:.2%} ({format_valid}/{total})")

metrics = {
    "model": MODEL_NAME,
    "adapter": ADAPTER_DIR,
    "questions": total,
    "full_validation_set": EVAL_LIMIT is None,
    "correct": correct,
    "accuracy": accuracy,
    "format_valid": format_valid,
    "format_valid_rate": format_valid_rate,
    "per_class": {
        label: {
            "correct": int(per_class.loc[label, "correct"]),
            "count": int(per_class.loc[label, "count"]),
            "accuracy": float(per_class.loc[label, "accuracy"]),
        }
        for label in labels
    },
    "confusion_matrix": {
        target: {
            predicted: int(confusion.loc[target, predicted])
            for predicted in confusion.columns
        }
        for target in confusion.index
    },
}

with open(
    "validation_predictions.jsonl",
    "w",
    encoding="utf-8",
) as handle:
    for row in predictions:
        handle.write(json.dumps(row, ensure_ascii=False) + "\n")

with open(
    "validation_metrics.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        metrics,
        handle,
        ensure_ascii=False,
        indent=2,
    )

print("Saved validation_predictions.jsonl and validation_metrics.json")

## 7. Save the adapter and evaluation artifacts

The resulting directory is a LoRA adapter, not a merged model. Later GRPO must load it over `unsloth/Qwen3.5-4B` with `fast_inference=False` and preserve the native `<think>...</think>\n\n\boxed{...}` contract. Do not reuse a Qwen3-4B-Base GRPO notebook or its custom `<start_working_out>` tags unchanged.


In [ ]:
import shutil

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

with open(Path(ADAPTER_DIR) / "training_handoff.json", "w", encoding="utf-8") as handle:
    json.dump(
        {
            "base_model": MODEL_NAME,
            "max_seq_length": MAX_SEQ_LENGTH,
            "load_in_4bit": False,
            "load_in_16bit": True,
            "fast_inference": False,
            "response_contract": "<think>...</think>\\n\\n\\boxed{candidate}",
            "grpo_note": (
                "Load this adapter over the same Qwen3.5 base and tokenizer. "
                "Keep native thinking tags and reward the strict final boxed label."
            ),
        },
        handle,
        indent=2,
    )

adapter_archive = shutil.make_archive(ADAPTER_DIR, "zip", root_dir=ADAPTER_DIR)
print("Adapter archive:", adapter_archive)
print("Evaluation files: validation_predictions.jsonl, validation_metrics.json")

try:
    from google.colab import files

    files.download(adapter_archive)
    files.download("validation_metrics.json")
    files.download("validation_predictions.jsonl")
except ImportError:
    print("Not running in Colab; artifacts remain in the current directory.")